# Tutorial — Seleção de Features via PySpark (`yggdrasil.feature_selection`)

Seleção de **features** para modelos de classificação/regressão, organizada por **books** (origens de
dados — ex.: serasa, bvs, cadastral, transacional, open finance). Para cada book, roda um pipeline de
filtros duros (missing, variância, redundância) + importância (RandomForest + univariadas) + **Boruta**,
consolidado num **consenso** (`selecionada` + `motivo`). No fim: tabela e painéis por book, um **ranking
global**, e um bloco de **análises visuais pós-seleção** para entender, justificar e auditar o conjunto
final antes de modelar. É um subpacote **isolado** — não interfere na esteira de modelo nem na EDA.

> **Spark-first.** O entrypoint recebe um **Spark DataFrame** com o contrato do pacote: features `feat_`,
> alvo `target` e (opcional) coluna de amostra `amostra` (a seleção usa a amostra de desenvolvimento `DES`).
> Requer o extra `[spark]`: `pip install 'yggdrasil[spark]'`.

**Roteiro:** 0) setup · 1) catálogo de books + dados sintéticos · 2) config · 3) quickstart · 4) relatório ·
5) books (3 formas) · 6) pipeline e `motivo` · **7) análises visuais pós-seleção** · 8) ajustes · 9) exportar.

## 0. Setup e sessão Spark

O notebook roda tanto no **Databricks** (onde `spark` já existe) quanto no **Jupyter local** (cria uma
`SparkSession` local). Em base grande, prefira o cluster e use `sample_size` (ver seção 8).

In [ ]:
# --- Bootstrap: torna o pacote `yggdrasil` importável a partir do repositório,
# sem `pip install`. Procura a raiz do repo (a pasta que contém `yggdrasil/`) por
# vários âncoras — o caminho do próprio notebook (VS Code expõe `__vsc_ipynb_file__`),
# o diretório atual, os diretórios do sys.path e, no Databricks, o caminho via
# dbutils — subindo até achá-la, e a insere no sys.path. Cobre Jupyter/VS Code
# local e Databricks; se o pacote já estiver importável, é inócuo.
import sys
from pathlib import Path

def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None

_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(7)

# Usa o `spark` do Databricks se existir; senão, cria uma sessão local.
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("ygg-feature-selection-tutorial")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 1. Catálogo de books e dicionário de features

Vamos montar uma base sintética de risco de crédito com **5 books** (origens de dados). Cada feature tem
um **papel didático** que exercita uma etapa do pipeline — dá para prever o `motivo` de cada seleção/descarte.

**Book `serasa`** — bureau de crédito

| feature | descrição de negócio | papel esperado |
|---|---|---|
| `feat_serasa_score` | Score de crédito do bureau (↑ = menor risco) | **forte preditora** |
| `feat_serasa_atraso_max` | Atraso máximo (dias) em 12m | **redundante** c/ o score (\|corr\|≈1) |
| `feat_serasa_qt_consultas_3m` | Nº de consultas ao CPF em 3m (apetite por crédito) | moderada |
| `feat_serasa_vlr_dividas` | Valor de dívidas negativadas | moderada |
| `feat_serasa_qt_negativacoes` | Nº de negativações ativas | fraca / ruído |
| `feat_serasa_const` | Campo legado sempre = 1 | **sem variância** → descartada |

**Book `bvs`** — bureau alternativo (Boa Vista)

| feature | descrição | papel |
|---|---|---|
| `feat_bvs_score_recuperacao` | Score de propensão a recuperação | moderada |
| `feat_bvs_qt_protestos` | Nº de protestos em cartório | moderada |
| `feat_bvs_util_limite` | Utilização do limite (%) | fraca/moderada |
| `feat_bvs_miss` | Indicador de fonte instável | **80% missing** → descartada |

**Book `cadastral`** — dados internos

| feature | descrição | papel |
|---|---|---|
| `feat_cadastral_renda` | Renda mensal comprovada | **forte preditora** |
| `feat_cadastral_idade` | Idade (anos) | fraca |
| `feat_cadastral_tempo_rel` | Tempo de relacionamento (meses) | moderada |
| `feat_cadastral_flag_pep` | Flag PEP (raríssima) | **quase-constante** → descartada |
| `feat_cadastral_uf` | UF de residência (string) | **não-numérica** (sem métricas de modelo) |

**Book `transacional`** — comportamento de conta/cartão (6m)

| feature | descrição | papel |
|---|---|---|
| `feat_transacional_saldo_medio` | Saldo médio em conta | moderada |
| `feat_transacional_pct_cheque` | % de dias usando cheque especial | moderada (estresse) |
| `feat_transacional_qt_saques` | Nº de saques em espécie | fraca |

**Book `openfinance`** — agregação Open Finance

| feature | descrição | papel |
|---|---|---|
| `feat_openfinance_renda_presumida` | Renda presumida por fluxo | **redundante cross-book** c/ `cadastral_renda` |
| `feat_openfinance_qt_vinculos` | Nº de instituições vinculadas | fraca |
| `feat_openfinance_vazamento` | Marcação gerada **após** a decisão | **leakage** → barrada |

> **Ponto-chave (cross-book).** A deduplicação de redundância do pipeline roda **por book**. Por isso
> `cadastral_renda` e `openfinance_renda_presumida` (books diferentes, mas ~0.9 correlacionadas) **ambas
> sobrevivem** — e só o heatmap de correlação dos **sobreviventes** (seção 7.10) revela essa redundância.

In [ ]:
n = 6000

# Fatores latentes (não entram na base; geram os sinais).
risco   = rng.normal(size=n)        # ↑ = pior crédito
renda_l = rng.normal(size=n)        # ↑ = maior renda (melhor)
comport = rng.normal(size=n)        # comportamento de conta

df = pd.DataFrame({
    # ── book serasa ────────────────────────────────────────────────
    "feat_serasa_score":         500 - 150 * risco + rng.normal(0, 30, n),      # forte (↑ = bom)
    "feat_serasa_atraso_max":    np.clip(30 * risco + rng.normal(0, 2, n), 0, None),  # ~ -score → redundante
    "feat_serasa_qt_consultas_3m": rng.poisson(2 + np.clip(risco, 0, None), n),
    "feat_serasa_vlr_dividas":   np.clip(2000 * risco + rng.normal(0, 500, n), 0, None),
    "feat_serasa_qt_negativacoes": rng.poisson(0.5, n),                          # ruído
    "feat_serasa_const":         1.0,                                            # sem variância
    # ── book bvs ───────────────────────────────────────────────────
    "feat_bvs_score_recuperacao": -0.5 * risco + rng.normal(0, 0.9, n),
    "feat_bvs_qt_protestos":     rng.poisson(0.3 + np.clip(risco, 0, None) * 0.5, n),
    "feat_bvs_util_limite":      np.clip(0.4 + 0.15 * risco + rng.normal(0, 0.2, n), 0, 1),
    "feat_bvs_miss":             rng.normal(size=n),                             # vira 80% missing
    # ── book cadastral ─────────────────────────────────────────────
    "feat_cadastral_renda":      np.clip(3000 + 1500 * renda_l + rng.normal(0, 400, n), 500, None),  # forte
    "feat_cadastral_idade":      np.clip(38 + 10 * rng.normal(size=n), 18, 80),  # fraca
    "feat_cadastral_tempo_rel":  np.clip(24 + 12 * comport + rng.normal(0, 6, n), 0, None),
    "feat_cadastral_flag_pep":   (rng.random(n) < 0.003).astype(int),           # quase-constante
    "feat_cadastral_uf":         rng.choice(["SP", "RJ", "MG", "RS", "BA"], n),  # não-numérica
    # ── book transacional ──────────────────────────────────────────
    "feat_transacional_saldo_medio": 1000 * renda_l + 500 * comport + rng.normal(0, 300, n),
    "feat_transacional_pct_cheque":  np.clip(0.2 + 0.15 * risco + rng.normal(0, 0.1, n), 0, 1),
    "feat_transacional_qt_saques":   rng.poisson(3, n),                         # fraca
    # ── book openfinance ───────────────────────────────────────────
    "feat_openfinance_qt_vinculos":  rng.poisson(2, n),                         # fraca
})

# Redundância CROSS-BOOK: renda presumida ~ 0.9 da renda cadastral (books diferentes).
df["feat_openfinance_renda_presumida"] = 0.9 * df["feat_cadastral_renda"] + rng.normal(0, 350, n)

# Alvo binário: pior risco / menor renda → maior prob. de default.
logit = 1.1 * risco - 0.9 * renda_l - 0.4 * comport + rng.normal(0, 0.6, n)
df["target"] = (logit > 0.4).astype(int)

# Leakage óbvio: cópia ruidosa do alvo (deve ser BARRADA pela flag de leakage).
df["feat_openfinance_vazamento"] = df["target"] + rng.normal(0, 0.01, n)

# 80% de missing em feat_bvs_miss (acima do missing_max default 50% → descarte).
df.loc[df.sample(frac=0.8, random_state=1).index, "feat_bvs_miss"] = np.nan

# Contrato: amostra (seleção usa só DES) e data de referência.
df["amostra"] = np.where(rng.random(n) < 0.7, "DES", "OOT")
df["dt_ref"]  = pd.Timestamp("2024-06-01")

sdf = spark.createDataFrame(df)
print("linhas:", sdf.count(), "| colunas de feature:",
      len([c for c in sdf.columns if c.startswith("feat_")]),
      "| taxa de evento:", round(df["target"].mean(), 3))
sdf.limit(5).toPandas()

## 2. Configuração

`ColumnConfig` define o **contrato de colunas** (prefixo `feat_`, `target`, `amostra`, `DES`).
`FeatureSelectionConfig` reúne os parâmetros da seleção. Usamos uma config **leve** para o tutorial rodar
rápido (Boruta com poucas iterações, floresta pequena). Os **defaults são mais pesados** — veja a
referência completa no `README.md` do módulo.

In [ ]:
from yggdrasil import ColumnConfig
from yggdrasil.feature_selection import run_feature_selection, FeatureSelectionConfig

cfg = ColumnConfig()   # feature_prefix="feat_", target_col="target", sample_col="amostra", dev_sample="DES"

fs_cfg = FeatureSelectionConfig(
    missing_max=0.50,        # descarta feature com >50% de missing
    corr_high=0.80,          # |corr| acima disso = redundância (POR BOOK)
    boruta_enable=True,      # liga o Boruta
    boruta_max_iter=12,      # leve p/ o tutorial (default 50)
    rf_n_estimators=60,      # floresta pequena p/ o tutorial (default 100)
    rf_max_depth=5,
    backend="spark",         # pyspark.ml (use "driver" p/ sklearn no driver)
    top_k_book=12,
    top_k_overall=15,
)
fs_cfg

## 3. Quickstart — uma chamada

`run_feature_selection` faz tudo: resolve os books, roda o pipeline por book e devolve um
`FeatureSelectionReport`. Passamos `books=["serasa", "bvs", "cadastral", "transacional", "openfinance"]`
(match por **palavra-chave**, ver seção 5).

> Em Spark local isso pode levar de alguns segundos a ~2 min (várias florestas + Boruta em 5 books).

In [ ]:
BOOKS = ["serasa", "bvs", "cadastral", "transacional", "openfinance"]

report = run_feature_selection(
    sdf, cfg, fs_cfg,
    books=BOOKS,
    problem_type="classification",   # explícito (poderia ser inferido do alvo)
    with_panels=True,                # monta TODOS os painéis (inclui os pós-seleção)
)
report.summary()   # por book: nº de features, selecionadas e descartadas

## 4. Lendo o relatório

O `FeatureSelectionReport` traz: `selection_table` (1 linha/feature), `book_tables`, `selected_features`
(dict por book), `selected_overall` (lista achatada), `overall_importance` (ranking global), `panels`
(figuras), `overall_correlation` (correlação **cross-book** das selecionadas — seção 7.10), `fs_cfg`/`cfg`
e `problem_type`.

In [ ]:
print("Selecionadas por book:")
for book, feats in report.selected_features.items():
    print(f"  {book:14s}: {feats}")
print("\nTotal selecionado (overall):", len(report.selected_overall))

In [ ]:
# Tabela de seleção (colunas-chave). 'motivo' explica cada decisão.
cols = ["book", "feature", "pct_missing", "sem_variancia", "score",
        "boruta_decisao", "score_consenso", "selecionada", "motivo"]
report.selection_table[cols]

In [ ]:
# Ranking GLOBAL das selecionadas (recalculado sobre todas as selecionadas juntas).
report.overall_importance.head(15)

## 5. Books — as 3 formas de definir

`books` aceita três formas. Use `resolve_books` para inspecionar o agrupamento **antes** de rodar a
seleção (ele só lê os nomes de coluna).

In [ ]:
from yggdrasil.feature_selection import resolve_books

print("(1) Lista de palavras-chave (match 'contains', case-insensitive):")
for b in resolve_books(sdf, cfg, ["serasa", "cadastral"]):
    print(f"    {b.name}: {b.features}")

print("\n(2) Auto (books=None) — 1º segmento após o prefixo 'feat_':")
for b in resolve_books(sdf, cfg, None):
    print(f"    {b.name}: {len(b.features)} features")

print("\n(3) Dict explícito — colunas escolhidas a dedo:")
books_dict = {"bureaus": ["feat_serasa_score", "feat_bvs_score_recuperacao"]}
for b in resolve_books(sdf, cfg, books_dict):
    print(f"    {b.name}: {b.features}")

## 6. O pipeline por book e os `motivo`

A ordem por book é: **missing → variância (P1==P99) → importância → redundância → Boruta → consenso**.
Cada descarte registra um `motivo`. Descartes **determinísticos** (filtros duros):

- `feat_serasa_const` → **`sem variância`**;
- `feat_cadastral_flag_pep` → **`quase-constante`**;
- `feat_bvs_miss` → **`alto missing`** (80% > 50%);
- `feat_openfinance_vazamento` → **`suspeita de leakage (revisar)`** (barrada mesmo com sinal altíssimo);
- `feat_serasa_score` × `feat_serasa_atraso_max` (mesmo book, \|corr\|≈1) → **`redundante`** (uma vira
  representante, a outra sai; qual sobrevive pode variar entre execuções).

In [ ]:
sel = report.selection_table.set_index("feature")
casos = ["feat_serasa_const", "feat_cadastral_flag_pep", "feat_bvs_miss",
         "feat_openfinance_vazamento", "feat_serasa_score", "feat_serasa_atraso_max"]
sel.loc[[f for f in casos if f in sel.index], ["book", "selecionada", "motivo"]]

In [ ]:
# Distribuição dos motivos (quem saiu/entrou e por quê).
report.selection_table["motivo"].value_counts(dropna=False)

## 7. Análises visuais pós-seleção

Este é o foco do tutorial: **depois** que a seleção rodou, o que ajuda a *entender, justificar e auditar*
o conjunto final antes de modelar. Com `with_panels=True`, tudo já vem pronto em `report.panels`; as
funções também são chamáveis direto via `from yggdrasil.feature_selection import plots`.

Cada figura responde uma pergunta prática:

| Painel | Pergunta que responde |
|---|---|
| **Dashboard** (7.1) | tudo numa tela |
| Funil (7.2) | *onde* perdi cada feature? |
| Mapa de decisão (7.3) | quais são âncoras e o que revisar? |
| Contribuição por book (7.4) | de onde vem o poder? dependo demais de uma fonte? |
| Redundância por cluster (7.5) | por que descartei X e mantive Y? |
| Boruta (7.6) | a evidência estatística é robusta? |
| Quadrante IV×KS (7.7) | poder preditivo por faixa (Siddiqi) |
| Auditoria de leakage (7.8) | algum sinal é bom demais para ser verdade? |
| Scorecard (7.9) | perfil de qualidade feature a feature |
| Correlação cross-book (7.10) | sobrou redundância entre books? |

### 7.1 Painel pós-seleção (dashboard 2×2)

Visão-resumo: funil + mapa de decisão + Boruta + contribuição por book. Ideal para colar num relatório
de validação de modelo.

In [ ]:
report.panels["post_selection_dashboard"]

### 7.2 Funil de seleção

Lê-se de cima (candidatas) para baixo (selecionadas): cada `−k` em crimson é quanto caiu naquela etapa,
derivado do `motivo`. Se uma etapa derruba demais (ex.: `redundância` levando >50% do atrito), é sinal de
limiar mal calibrado (`corr_high`, `missing_max`) para re-rodar. A barra final é o tamanho do conjunto
que vai ao modelo.

In [ ]:
report.panels["funnel"]

### 7.3 Mapa de decisão

`x` = importância multivariada (RandomForest), `y` = sinal univariado (KS). Azul = selecionada, cinza =
descartada, estrela crimson = leakage; o tamanho ∝ consenso. Quadrante alto-alto = **âncoras**; alto-`x`
+ `y`≈0 = suspeita de interação/leakage a revisar; descartada de alto sinal = confira o `motivo`
(saiu por redundância/leakage/consenso, não por acaso).

In [ ]:
report.panels["decision_map"]

### 7.4 Contribuição por book (concentração)

Barra escura = % da importância total (Σ `rf_importance`); barra clara = % das features. O **HHI** no
título mede concentração: um único book com share alto = risco de dependência de uma fonte de dados
(continuidade/custo de ingestão, e um ponto de atenção sob Basileia).

In [ ]:
report.panels["book_power"]

### 7.5 Redundância por cluster

Para cada cluster de features correlacionadas (dentro de um book), mostra a **representante** mantida (★,
a de maior importância) e as **redundantes** descartadas (`→ representante`). Documenta a poda de
multicolinearidade sem precisar abrir a matriz de correlação.

In [ ]:
report.panels["cluster_redundancy"]

### 7.6 Boruta — hits × bandas de significância

Lollipop dos `hits` por feature. Sob H0, `hits ~ Binomial(n_iter, 0.5)`; as linhas tracejadas são os
limiares de **confirmação/rejeição** com correção de Bonferroni, e a faixa cinza é a zona de *tentativa*.
Cor do ponto = decisão do Boruta; o **anel** marca features que entraram pelo consenso **apesar** de o
Boruta ter rejeitado (revisar à mão). Poucas iterações (aqui `n_iter=12`) dão bandas exigentes — no
default (`50`) a separação fica mais nítida.

In [ ]:
report.panels["boruta"]

### 7.7 Quadrante de poder preditivo (IV × KS)

Só em classificação. Faixas de **Information Value** (Siddiqi): <0.02 inútil · 0.02–0.1 fraco · 0.1–0.3
médio · 0.3–0.5 forte · >0.5 **zona de leakage** (sombreada). A linha horizontal é o `ks_min` de
referência. Útil para bater o olho em quem tem poder de fato e quem é bom demais para ser verdade.

In [ ]:
report.panels["power_quadrant"]

### 7.8 Auditoria de leakage

Watchlist das features com maior sinal univariado (Gini). As **flagradas** por leakage aparecem em crimson
e anotadas `⚠ barrada`; a linha tracejada é o limiar de leakage. `feat_openfinance_vazamento` (cópia do
alvo) deve saltar no topo, muito além do limiar.

In [ ]:
report.panels["leakage_audit"]

### 7.9 Scorecard de qualidade dos sobreviventes

Folha de auditoria: uma linha por feature **selecionada**, colunas = métricas de qualidade normalizadas
(azul = bom, crimson = ruim). Missing e dominância são invertidos (menos é melhor). Flagra num relance
sinais de instabilidade (missing alto, categoria dominante, consenso só no limiar). Cinza = métrica
ausente (ex.: `feat_cadastral_uf`, não-numérica).

In [ ]:
report.panels["survivor_scorecard"]

### 7.10 Correlação cross-book dos sobreviventes

**O ponto-chave.** A deduplicação do pipeline é **por book**, então redundância entre books escapa. Aqui
o heatmap Spearman é do conjunto **final** (todas as selecionadas, atravessando books); pares com
`|corr| ≥ corr_high` ficam **contornados em crimson**. `feat_cadastral_renda` e
`feat_openfinance_renda_presumida` (books diferentes, ~0.9) devem aparecer contornadas — candidatas a um
corte manual antes de modelar.

In [ ]:
# vem pronto em report.panels quando há >= 2 selecionadas numéricas...
fig = report.panels.get("survivor_correlation")
# ...e a matriz crua fica em report.overall_correlation (Spearman, cross-book):
print("features na matriz de sobreviventes:", list(report.overall_correlation.columns))
fig if fig is not None else "menos de 2 selecionadas numéricas — sem heatmap"

In [ ]:
# Chamando um gráfico DIRETO (fora de report.panels), com os limiares do fs_cfg:
from yggdrasil.feature_selection import plots
plots.plot_boruta_significance(report.selection_table,
                               n_iter=report.fs_cfg.boruta_max_iter,
                               alpha=report.fs_cfg.boruta_alpha)

## 8. Ajustes: Boruta, backend e amostragem

- **`boruta_enable=False`** desliga a etapa mais cara (a seleção passa a se apoiar em importância +
  consenso). Útil para iterar rápido.
- **`backend="driver"`** roda RandomForest/Boruta com `sklearn` no driver (amostra via `toPandas()`) —
  bom para bases pequenas; **`"spark"`** escala no cluster.
- **`sample_size > 0`** amostra `N` linhas **só para as etapas de modelo** (RF e Boruta); a correlação e
  as univariadas usam sempre o dataset completo.

In [ ]:
fs_rapido = FeatureSelectionConfig(boruta_enable=False, rf_n_estimators=60, rf_max_depth=5,
                                   top_k_book=12, top_k_overall=15)
rep2 = run_feature_selection(sdf, cfg, fs_rapido, books=BOOKS,
                             problem_type="classification", with_panels=False)
print("Com Boruta:", sorted(report.selected_overall))
print("Sem Boruta:", sorted(rep2.selected_overall))

## 9. Exportar e logar

`to_csv` grava a `selection_table`; `to_html(embed_panels=True)` gera um relatório com **todos** os
painéis embutidos (PNG base64) — inclusive os pós-seleção. Para registrar no MLflow, passe
`mlflow_experiment=...` em `run_feature_selection`.

> **Databricks:** aponte os caminhos para DBFS/Volumes (ex.: `/dbfs/tmp/selecao_features.csv`); no
> Jupyter local, o diretório atual já serve.

In [ ]:
report.to_csv("selecao_features.csv")
with open("relatorio_selecao.html", "w", encoding="utf-8") as fh:
    fh.write(report.to_html(embed_panels=True))
print("Arquivos gravados: selecao_features.csv, relatorio_selecao.html")

In [ ]:
# MLflow (descomente em um ambiente com tracking, ex.: Databricks):
# report_mlf = run_feature_selection(
#     sdf, cfg, fs_cfg, books=BOOKS,
#     mlflow_experiment="/Shared/feature_selection",   # loga tabela, html, painéis e métricas
#     run_name="fs_5books_v1",
# )

## 10. Notas

- **Numeric-only:** RF, correlação, IV/KS/AUC/Gini e Boruta operam só em colunas **numéricas**;
  não-numéricas (ex.: `feat_cadastral_uf`) ficam `NaN` nesses indicadores e entram como representantes.
- **Classificação × regressão:** `iv/ks/auc/gini` existem **só em classificação**; em regressão o
  `score`/consenso se apoiam em `rf_importance` e `corr_target`, e o quadrante IV×KS degrada para uma
  mensagem. O mapa de decisão e a auditoria de leakage usam `|corr_target|` como fallback.
- **Consenso:** `leakage_flag` **barra** a seleção (prioridade máxima); Boruta `confirmada` seleciona;
  senão a feature entra pelo `score_consenso >= consensus_threshold`. Boruta `rejeitada` **não impede** —
  se o consenso atingir o limiar, entra assim mesmo (o **anel** no gráfico 7.6 marca esses casos).
- **Cross-book:** a redundância é podada **por book**; a correlação dos sobreviventes (7.10) é onde a
  redundância entre books aparece — sempre confira antes de modelar.
- **Referência completa:** `yggdrasil/feature_selection/README.md`.
- Outros tutoriais: `00_tutorial_yggdrasil.ipynb` (classificação), `01_tutorial_lgd.ipynb` (regressão),
  `02_tutorial_eda_features.ipynb` (EDA).